# Introduction to PySpark

Example notebook to verify that the environment is working correctly and to learn the basic PySpark operations.

## 1. Environment check

First, we verify that Python and PySpark are installed correctly in the container. If the following cells print the versions without errors, the environment is ready.

In the first cell all the libraries used throughout the notebook are imported. It is good practice to put all imports at the top, so it is immediately clear what the code depends on.

In [ ]:
import os
import sys
from pathlib import Path

import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import expr
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType

In [ ]:
BASE_FILEPATH = Path("/home/jovyan/data")

In [ ]:
print(f"Python: {sys.version}")
print(f"PySpark: {pyspark.__version__}")

## 2. Creating a SparkSession

The `SparkSession` is the entry point for working with PySpark. Every Spark program needs one: it is the object used to create DataFrames, run SQL queries and configure Spark's behavior.

- **`appName`** assigns a name to the session (visible in the Spark UI)
- **`getOrCreate`** reuses an existing session if one is available, otherwise creates a new one

In [ ]:
appName = "IntroSparkSession"

spark = SparkSession.builder \
    .appName(appName) \
    .getOrCreate()

spark_ui_port = os.environ.get("SPARK_UI_PORT", "4040")
print(f"Spark UI: http://localhost:{spark_ui_port}")
print(f"Version: {pyspark.__version__}")
print(f"Master: local[*]")
print(f"AppName: {appName}")

## 3. Creating a DataFrame with an explicit schema

The schema is defined using `StructType` and `StructField`. This approach is preferable to letting Spark infer the types automatically, because it gives full control over data types and avoids silent errors.

The dataset represents a small list of employees in an Italian company, with name, city, age, annual salary and department.

In [ ]:
schema = StructType(fields=[
    StructField(name="name", dataType=StringType(), nullable=False),
    StructField(name="city", dataType=StringType(), nullable=False),
    StructField(name="age", dataType=IntegerType(), nullable=False),
    StructField(name="salary", dataType=FloatType(), nullable=False),
    StructField(name="department", dataType=StringType(), nullable=False),
])

data = [
    ("Marco", "Roma", 30, 32_000.0, "Sales"),
    ("Giulia", "Milano", 28, 35_000.0, "IT"),
    ("Luca", "Napoli", 35, 28_000.0, "Sales"),
    ("Sara", "Torino", 26, 31_000.0, "Marketing"),
    ("Andrea", "Roma", 32, 40_000.0, "IT"),
    ("Chiara", "Milano", 29, 37_000.0, "IT"),
    ("Davide", "Napoli", 31, 29_000.0, "Marketing"),
    ("Francesca", "Torino", 27, 33_000.0, "Sales"),
    ("Paolo", "Roma", 45, 52_000.0, "IT"),
    ("Elena", "Milano", 33, 34_000.0, "Sales"),
    ("Roberto", "Napoli", 38, 41_000.0, "Marketing"),
    ("Valentina", "Torino", 24, 26_000.0, "IT"),
]

df = spark.createDataFrame(data, schema)
df.show()

### Inspecting a DataFrame

`df.columns` returns the list of column names as plain Python strings. `printSchema()` shows the full structure including data types and nullability. These are typically the first commands to run when working with a new DataFrame:

In [ ]:
df.columns

In [ ]:
df.printSchema()

`describe()` automatically computes summary statistics for numeric columns (count, mean, standard deviation, min and max). It is useful for a quick overview of the data:

In [ ]:
df.describe().show()

### Extracting data from a DataFrame

PySpark DataFrames are distributed — the data lives across the cluster, not in local memory. To bring data back to the driver (the local Python process) there are several methods:

- **`take(n)`** — returns the first `n` rows as a list of `Row` objects (safe, bounded)
- **`tail(n)`** — returns the last `n` rows
- **`collect()`** — returns **all** rows — use with caution on large datasets, as it loads everything into memory
- **`toPandas()`** — converts to a Pandas DataFrame — same memory warning as `collect()`

In [ ]:
first_rows = df.take(num=5)
first_rows

In [ ]:
last_rows = df.tail(num=5)
last_rows

In [ ]:
df_pandas = df.toPandas()
df_pandas.head()

## 4. Basic operations

PySpark provides chainable methods to transform DataFrames. The main operations are:

- **`select`** — select specific columns
- **`filter`** — select rows that satisfy a condition
- **`groupBy`** — group rows by one or more columns (followed by an aggregation like `count`, `avg`, `sum`)
- **`orderBy`** — sort rows by one or more columns
- **`withColumn`** — add a column or overwrite an existing one
- **`withColumnRenamed`** — rename a column
- **`drop`** — remove one or more columns
- **`dropDuplicates`** — remove duplicate rows

### select and filter

`select` is used to pick only the desired columns:

In [ ]:
df.select("name", "city").show()

`filter` is used to select rows that satisfy a condition. Here we filter employees with a salary above 35,000:

In [ ]:
df.filter(condition=df.salary > 35_000).show()

### groupBy and aggregations

`groupBy` groups rows by one or more columns. Here we count how many people are in each city:

In [ ]:
df.groupBy("city").count().show()

With `agg()` multiple aggregations can be computed at once. Here we calculate for each department: the number of employees, the average salary, the minimum and the maximum:

In [ ]:
df.groupBy("department").agg(
    F.count("name").alias("num_employees"),
    F.round(F.avg("salary"), 2).alias("avg_salary"),
    F.min("salary").alias("min_salary"),
    F.max("salary").alias("max_salary"),
).show()

### orderBy

`orderBy` sorts rows by one or more columns. Here we sort by salary in descending order:

In [ ]:
df.orderBy(df.salary.desc()).show()

### withColumn and withColumnRenamed

`withColumn` creates a new column (or overwrites an existing one) from an expression. Here we calculate a 10% bonus on the salary:

In [ ]:
df_with_bonus = df.withColumn(
    colName="bonus",
    col=F.round(df.salary * 0.1, scale=2),
)

df_with_bonus.show()

`withColumnRenamed` renames an existing column. Useful when column names in the original dataset are unclear or contain spaces:

In [ ]:
df_with_bonus \
    .withColumnRenamed(existing="city", new="location") \
    .withColumnRenamed(existing="salary", new="annual_salary") \
    .show()

### when / otherwise (conditional logic)

`when` and `otherwise` work like a `CASE WHEN` in SQL: they assign different values based on a condition. Here we classify employees into salary brackets:

In [ ]:
df_with_bracket = df.withColumn(
    colName="salary_bracket",
    col=F.when(condition=df.salary < 30_000, value="Low")
        .when(condition=df.salary < 38_000, value="Medium")
        .otherwise(value="High"),
)

df_with_bracket.show()

### drop and dropDuplicates

`drop` removes one or more columns from the DataFrame:

In [ ]:
df.drop("age", "salary").show()

`dropDuplicates` removes duplicate rows. If a list of columns is passed, only those columns are considered for duplicates (keeping the first occurrence). Here we extract the unique cities in the dataset:

In [ ]:
df.select("city").dropDuplicates().show()

## 5. Joining DataFrames

A **join** combines two DataFrames based on a common column, just like a `JOIN` in SQL.

We create a second DataFrame with department information. Note: the `HR` department has no employees in our dataset — this will help demonstrate the difference between join types.

In [ ]:
dept_schema = StructType(fields=[
    StructField(name="department", dataType=StringType(), nullable=False),
    StructField(name="headquarters", dataType=StringType(), nullable=False),
    StructField(name="budget", dataType=FloatType(), nullable=False),
])

dept_data = [
    ("IT", "Milano", 200_000.0),
    ("Sales", "Roma", 150_000.0),
    ("Marketing", "Torino", 120_000.0),
    ("HR", "Roma", 80_000.0),
]

df_dept = spark.createDataFrame(data=dept_data, schema=dept_schema)
df_dept.show()

**Inner join**: returns only the rows that have a match in both DataFrames. HR does not appear because no employee belongs to that department:

In [ ]:
df_inner = df.join(other=df_dept, on="department", how="inner")
df_inner.show()

**Left join**: keeps all rows from the left DataFrame (`df`), even if they have no match on the right. In this case the result is the same as the inner join because all employees have a valid department:

In [ ]:
df_left = df.join(other=df_dept, on="department", how="left")
df_left.show()

**Right join**: keeps all rows from the right DataFrame (`df_dept`). Now HR appears with `null` values for the employee columns, because it has no matches:

In [ ]:
df_right = df.join(other=df_dept, on="department", how="right")
df_right.show()

## 6. Spark SQL

In addition to the DataFrame API, PySpark allows writing queries in **pure SQL**. To do this, the DataFrame must first be registered as a temporary view with `createOrReplaceTempView`, then `spark.sql()` is used to execute the query.

In [ ]:
df.createOrReplaceTempView(name="employees")

sql_query = """SELECT department,
       COUNT(*) AS num_employees,
       ROUND(AVG(salary), 2) AS avg_salary,
       MIN(salary) AS min_salary,
       MAX(salary) AS max_salary
FROM employees
GROUP BY department
ORDER BY avg_salary DESC"""

result = spark.sql(sqlQuery=sql_query)
result.show()

### Mixing SQL and DataFrame API with `expr` and `selectExpr`

Sometimes it is convenient to use SQL syntax inside the DataFrame API. `expr()` lets you write a SQL expression as a column, while `selectExpr()` lets you select columns using SQL syntax directly:

In [ ]:
df.select("name", "salary", expr("salary * 0.1 AS bonus")).show()

In [ ]:
df.selectExpr(
    "name",
    "UPPER(city) AS city_upper",
    "salary",
    "CASE WHEN salary >= 38000 THEN 'High' ELSE 'Standard' END AS level",
).show()

### Registering a UDF for SQL

A **UDF** (User Defined Function) is a custom Python function that can be used inside SQL queries. First the function is registered with `spark.udf.register`, then it can be called by name in any `spark.sql()` query:

In [ ]:
def salary_after_tax(salary: float) -> float:
    """Compute the net salary.

    Parameters
    ----------
    salary : float
        The gross salary.

    Returns
    -------
    float
        The net salary.
    """
    tax_rate = 0.30
    return salary * (1 - tax_rate)

In [ ]:
spark.udf.register(name="salary_after_tax", f=salary_after_tax, returnType=FloatType())

sql_query = """SELECT name,
       salary,
       ROUND(salary_after_tax(salary), 2) AS net_salary
FROM employees
ORDER BY net_salary DESC
"""

spark.sql(sqlQuery=sql_query).show()

## 7. File I/O (CSV, Parquet and ORC)

PySpark can read and write data in several formats. The most common are:

- **CSV** — text format, readable by any program, but slow and without data type information
- **Parquet** — binary columnar format, standard in the Spark/Big Data world. Very efficient for reading and writing, and stores the schema inside the file
- **ORC** — another binary columnar format (Optimized Row Columnar), common in the Hive/Hadoop ecosystem. Similar to Parquet in performance

In this section we try the write/read cycle with all three formats.

### CSV

The DataFrame is written as CSV to the `data/` folder. The file will also be visible in the local filesystem thanks to the bind mount configured in `docker-compose.yml`:

In [ ]:
OUTPUT_FILEPATH = BASE_FILEPATH / "output"

In [ ]:
csv_path = str(OUTPUT_FILEPATH / "example_output_csv")

df.write \
    .mode("overwrite") \
    .option(key="header", value=True) \
    .csv(path=csv_path)

print(f"File written to: {csv_path}")

The CSV is read back to verify that the data was saved correctly:

In [ ]:
df_csv = spark.read \
    .option(key="header", value=True) \
    .option(key="inferSchema", value=True) \
    .csv(path=csv_path)

df_csv.printSchema()
df_csv.show()

### Parquet

The same DataFrame is written in Parquet format. Unlike CSV, Parquet automatically saves the schema (data types), so when reading back there is no need to specify `inferSchema` or `header`:

In [ ]:
parquet_path = str(OUTPUT_FILEPATH / "example_output_parquet")

df.write \
    .mode("overwrite") \
    .parquet(path=parquet_path)

print(f"File written to: {parquet_path}")

The Parquet file is read back. Notice that the schema is reconstructed automatically with the correct types, without any additional options:

In [ ]:
df_parquet = spark.read.parquet(parquet_path)

df_parquet.printSchema()
df_parquet.show()

### ORC

ORC (Optimized Row Columnar) is another binary columnar format, widely used in the Hadoop ecosystem. Like Parquet, it stores the schema and is efficient for large datasets:

In [ ]:
orc_path = str(OUTPUT_FILEPATH / "example_output_orc")

df.write \
    .mode("overwrite") \
    .orc(path=orc_path)

print(f"File written to: {orc_path}")

In [ ]:
df_orc = spark.read.orc(orc_path)

df_orc.printSchema()
df_orc.show()

## 8. All done!

If you made it here without errors, your PySpark environment is configured correctly.

**Next steps:**
- Load your own data into the `data/` folder and read it with `spark.read`
- Try using Spark SQL for more complex analyses
- Add libraries to `requirements.txt` and rebuild with `docker compose up --build`
- Monitor jobs in the [Spark UI](http://localhost:4040)

When done working, close the SparkSession to release resources:

In [ ]:
spark.stop()